Instalimi dhe importimi i librarive të nevojshme

In [ ]:
pip install pandas scipy seaborn matplotlib

In [58]:
import pandas as pd
import warnings
from scipy.stats import zscore
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

warnings.simplefilter(action='ignore', category=FutureWarning)

Mbledhja e të dhënave, definimi i tipeve të dhënave, kualiteti i të dhënave

In [ ]:
df = pd.read_csv(r'./master.csv')

print(df.dtypes)
print(df.describe())

Riemërimi i kolonave për përdorim më të lehtë

In [60]:
df=df.rename(columns={'sex':'gender','gdp_per_capita ($)':'gdp_per_capita',' gdp_for_year ($) ':'gdp_for_year', 'HDI for year' : 'hdi_for_year'})

Modifikimi i tipit të dhënave për kolonën 'gdp_for_year'

In [61]:
df['gdp_for_year'] = df['gdp_for_year'].str.replace(',', '').astype(int)

Ndryshimi i dimensionalitetit

In [62]:
df = df[['country','year', 'gender', 'age', 'suicides_no','population','suicides/100k pop','gdp_for_year','gdp_per_capita','hdi_for_year']]

Menaxhimi i vlerave null

In [63]:
df = df.sort_values(by=['country', 'year'])

df['hdi_for_year'] = df.groupby('country')['hdi_for_year'].transform(lambda x: x.fillna(method='ffill').fillna(method='bfill'))

yearly_mean = df.groupby(['country', 'year'])['hdi_for_year'].mean().reset_index()

yearly_mean['hdi_for_year'] = yearly_mean.groupby('country')['hdi_for_year'].transform(lambda x: x.interpolate(method='linear'))

df = df.merge(yearly_mean, on=['country', 'year'], suffixes=('', '_mean'))

df['hdi_for_year'] = df['hdi_for_year'].combine_first(df['hdi_for_year_mean'])

df = df.drop(columns=['hdi_for_year_mean'])

Mostrimi i të dhënave (10%)

In [ ]:
sampled_data = df.sample(frac=0.1)
print(sampled_data)

Validimi i vlerave duplikate <br>
Kontrollimi i kolonave specifike

In [ ]:
duplicates_check=['country','year','gender','age']
duplicates=df.duplicated(subset=duplicates_check)

if duplicates.any():
    print("Duplicates found. Dropping duplicates.")
    df = df.drop_duplicates()
    print("\nCleaned DataFrame:")
    print(df)
else:
    print("No duplicates found. DataFrame remains unchanged.")

Kontrollimi i tërë dataframe-it


In [ ]:
duplicates = df.duplicated()

if duplicates.any():
    print("Duplicates found.")
else:
    print("No duplicates found.")

Transformimi i kolonave specifike

In [67]:
df['total_suicides_year'] = df.groupby(["country", "year"])['suicides_no'].transform('sum')
df['total_population_year'] = df.groupby(["country", "year"])['population'].transform('sum')

df['suicides_to_population_ratio'] = df['suicides_no'] / df['population']

Diskretizimi i perpjestimit të vetëvrasjeve me numrin e popullesisë dhe i gdp në kategori më të përshtatshme

In [68]:
ratio_bins = [-1, 0, 1e-05, 2e-05, 4e-05, 6e-05, 8e-05, float('inf')]
ratio_labels = ['None','Very Low', 'Low', 'Medium', 'High', 'Very High', 'Extreme']

df['suicides_to_population_ratio_discretize'] = pd.cut(df['suicides_to_population_ratio'], bins=ratio_bins, labels=ratio_labels)

gdp_bins = [0, 1000, 2000, float('inf')]
gdp_labels = ['Low', 'Medium', 'High']
df['gdp_category'] = pd.cut(df['gdp_per_capita'], bins=gdp_bins, labels=gdp_labels)

Binarizimi i kolones 'gender'

In [69]:
df['gender_encoded'] = df['gender'].map({'male': 1, 'female': 0})

Ruajtja e transformimeve ne nje file te ri

In [70]:
df.to_csv('cleaned_data.csv', index=False)

Outlires Detection

In [ ]:
df=pd.read_csv(r'./cleaned_data.csv')


numerical_columns_country_year = ["suicides_no", "population", "suicides_to_population_ratio"]

# Group by the context
grouped_country_year = df.groupby(["country", "year"])

# Calculate z-scores within each group
def calculate_z_scores(group, numerical_columns):
    for col in numerical_columns:
        group[f"{col}_zscore"] = zscore(group[col]) if group[col].std() != 0 else 0
    return group

df = grouped_country_year.apply(calculate_z_scores, numerical_columns_country_year)

# Un-group for final result
df.reset_index(drop=True, inplace=True)


numerical_columns_country = ["gdp_for_year", "gdp_per_capita", "hdi_for_year"]

grouped_country = df.groupby(["country"])
df = grouped_country.apply(calculate_z_scores, numerical_columns_country)

# Un-group for final result
df.reset_index(drop=True, inplace=True)

# Identify outliers based on the z-score threshold
z_score_threshold = 3

outliers = df[
    (df[[f"{col}_zscore" for col in numerical_columns_country_year + numerical_columns_country]]
     .abs() > z_score_threshold).any(axis=1)
]

outliers_table = (df[
    [f"{col}_zscore" for col in numerical_columns_country_year + numerical_columns_country]
].abs() > z_score_threshold)

Display Outliers

In [ ]:
print(f"Number of outliers:", outliers.shape[0])  # Number of outlier rows
print(f"Outliers\n", outliers)
columns_with_outliers = [
    col.removesuffix('_zscore') 
    for col in outliers_table[[f"{col}" for col in outliers_table.columns]].any()[lambda x: x].index.tolist()
]

print(columns_with_outliers)

Fshirja e rreshtave me outliers dhe ruajtja e rezultatit ne file te ri

In [74]:
df = df.drop(outliers.index)
df.reset_index(drop=True, inplace=True)

df.to_csv('cleaned_data_without_outliers.csv', index=False)

df=pd.read_csv(r'./cleaned_data_without_outliers.csv')


Korelacioni i Vlerave Numerike

In [ ]:
correlation = df[numerical_columns_country_year + numerical_columns_country].corr()

sns.heatmap(correlation, annot=True, cmap='Blues')
plt.title('Correlation Matrix')

Eksplorimi i Relacioneve Multivariante në të Dhënat e Vetëvrasjeve dhe Popullsisë me Pairplot

In [ ]:
# Select numeric columns for the pairplot
numeric_columns = ['gdp_per_capita', 'total_suicides_year', 'suicides/100k pop', 'total_population_year']

# Create the pairplot
sns.pairplot(df[numeric_columns])
plt.show()

Shperndarja e popullsise pas log-transformimit

In [ ]:
df['log_population'] = np.log(df['population'] + 1)

sns.histplot(df['log_population'], bins=20, kde=True)
plt.title('Log-Transformed Distribution of Population')
plt.xlabel('Log(Population)')
plt.ylabel('Frequency')
plt.show()

Shperndarja e rasteve ne base te grupe-moshave

In [ ]:
age_suicidal_no = df.groupby('age')['suicides_no'].sum()

# Plot the aggregated data
age_suicidal_no.plot(kind='bar')
plt.xlabel('Age groups')
plt.ylabel('Suicide Number')
plt.title('Suicide Number per age group')
plt.show()